In [ ]:
import pandas as pd
from pathlib import Path
from time import time

# Normalization of Datasets

| DeviceID | Latitude | Longitude | Timestamp | Other features |
|:--------:|:--------:|:---------:|:---------:|:--------------:|
|  *String*  |   *Float*  |   *Float*   | *String*  | *...* |
|  **id**  |   **la**  |   **lon**   | **YYYY-MM-DD-HH:MM:SS**  | **...** |
|     0    |   39.9   |   120.5   | 2018-05-06-18:11:1 | ... |
|     0    |   40.1   |   120.7   | 2018-05-06-18:30:52 | ... |
|     1    |   29.4   |    99.5   | 2018-05-06-20:37:51 | ... |



In [3]:
normalized_datatypes = {
    'DeviceID': 'str',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'Timestamp': 'str',
}

## Shenzhen Urban

In [3]:
path_to_dataset_file = Path(r"Raw_Datasets/shenzhenurban_phonedata.csv")

column_names_type = {"SIMCardID": 'uint32',
                     "Time": 'str',
                     "Longitude": 'float64',
                     "Latitude": 'float64'
                     }
shenzhen_urban_df = pd.read_csv(path_to_dataset_file,
                 names = column_names_type.keys(), dtype = column_names_type)
shenzhen_urban_df

,SIMCardID,Time,Longitude,Latitude
0,55555556,17:39:08,113.809028,22.756181
1,55555556,23:53:07,114.026389,22.626042
2,55555556,13:10:09,114.039792,22.574028
3,55555556,16:08:30,113.805694,22.761389
4,55555556,20:50:17,113.846597,22.772500
...,...,...,...,...
38218712,55969825,20:09:51,114.042708,22.524583
38218713,55969825,20:08:41,114.042708,22.524583
38218714,55969825,18:44:44,114.045069,22.525139
38218715,55969825,18:44:15,114.045069,22.525139


In [ ]:
import datetime

def compute_shenzhensim_timestamp(df):
    df = df.copy()

    # Convert 'Time' to datetime, anchored at 1900-01-01
    df["Timestamp"] = pd.to_datetime(df["Time"], format="%H:%M:%S")

    # Sort by user ID only to preserve local order
    df.sort_values(by=["SIMCardID"], inplace=True)

    # Get previous time per user
    df["prev_time"] = df.groupby("SIMCardID")["Timestamp"].shift(1)

    # New day when time goes backward
    df["time_diff"] = (df["Timestamp"] < df["prev_time"]).astype(int)

    # Cumulative count of new days per user
    df["day_offset"] = df.groupby("SIMCardID")["time_diff"].cumsum()

    # Add day offset to base timestamp
    df["Timestamp"] = df["Timestamp"] + pd.to_timedelta(df["day_offset"], unit="D")

    # Format as string if needed
    df["Timestamp"] = df["Timestamp"].dt.strftime("%Y-%m-%d-%H:%M:%S")

    # Drop helper columns
    df.drop(columns=["prev_time", "time_diff", "day_offset"], inplace=True)

    return df


df = compute_shenzhensim_timestamp(shenzhen_urban_df)

,SIMCardID,Time,Longitude,Latitude,Timestamp
0,55555556,17:39:08,113.809028,22.756181,1900-01-01-17:39:08
1,55555556,23:53:07,114.026389,22.626042,1900-01-01-23:53:07
2,55555556,13:10:09,114.039792,22.574028,1900-01-02-13:10:09
4,55555556,20:50:17,113.846597,22.772500,1900-01-02-20:50:17
3,55555556,16:08:30,113.805694,22.761389,1900-01-03-16:08:30
...,...,...,...,...,...
38218713,55969825,20:08:41,114.042708,22.524583,1900-01-02-20:08:41
38218710,55969825,18:43:20,114.045069,22.525139,1900-01-03-18:43:20
38218709,55969825,18:43:46,114.045069,22.525139,1900-01-03-18:43:46
38218712,55969825,20:09:51,114.042708,22.524583,1900-01-03-20:09:51


In [ ]:
def shenzhenurban_normalize(df):
    column_names_remapper = {
        "SIMCardID": "DeviceID",
        "Latitude": "Latitude",
        "Longitude": "Longitude",
        "Timestamp": "Timestamp",
    }

    # If compute_shenzhensim_timestamp is needed, apply here
    # df = compute_shenzhensim_timestamp(df)

    df = df.rename(columns=column_names_remapper)

    # Instead of casting all at once, cast columns individually
    df['DeviceID'] = df['DeviceID'].astype('int32')
    df['Latitude'] = df['Latitude'].astype('float32')
    df['Longitude'] = df['Longitude'].astype('float32')
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

    return df

start = time()
shenzhen_urban_df_normalized = shenzhenurban_normalize(df)
print("Done in", time() - start, "seconds")


Done in 112.32550859451294 seconds


In [9]:
SAVE_PATH_SHENZHENURBAN = Path(r"PreprocessedData/ShenzhenUrban") / 'shenzhenurban_normalizedcols.pkl'

begin = time()
shenzhen_urban_df_normalized.to_pickle(path = SAVE_PATH_SHENZHENURBAN)
end = time()
print(f"Saving duration: {(end-begin):.2f} seconds.")

Saving duration: 10.41 seconds.


In [21]:
shenzhen_df = pd.read_pickle('PreprocessedData/0NormalizedCols/shenzhenurban.pkl')
shenzhen_df.to_csv('PreprocessedData/0NormalizedCols/shenzhenurban.csv', index=False)

## YJMob100K

In [11]:
path_to_folder = Path(r"Raw_Datasets/YJMob100K")
col_dtypes = {
    'uid': 'uint32',
    'd': 'uint8',
    't': 'uint8',
    'x': 'uint8',
    'y': 'uint8',
}

# dataset1 is the full dataset while dataset2 is a subset
yjmob_df = pd.read_csv(path_to_folder / 'yjmob100k-dataset1.csv', dtype=col_dtypes)
yjmob_df

,uid,d,t,x,y
0,0,0,1,79,86
1,0,0,2,79,86
2,0,0,8,77,86
3,0,0,9,77,86
4,0,0,19,81,89
...,...,...,...,...,...
111535170,99999,74,38,119,77
111535171,99999,74,39,132,94
111535172,99999,74,40,124,105
111535173,99999,74,41,121,107


In [12]:
import datetime

def compute_yjmob_timestamp(yjmob_datfra):
    USERID = 'uid'
    TIME = 't'
    DAY = 'd'
    FIRST_DAY = datetime.datetime.strptime('1900-01-01', '%Y-%m-%d')
    
    time_df = yjmob_datfra[[USERID, DAY, TIME]].copy()
    time_df[TIME] = time_df[TIME].apply(lambda x: f"{x // 2:02d}:{(x % 2)*30:02d}:00")
    time_df[DAY] = time_df[DAY].apply(lambda x: (FIRST_DAY + datetime.timedelta(days=x)).strftime('%Y-%m-%d'))
    
    yjmob_datfra['Timestamp'] = time_df[DAY] + '-' + time_df[TIME]
    
    return yjmob_datfra

df = compute_yjmob_timestamp(yjmob_df.head(10**6))
df

/tmp/ipykernel_1005848/1702095083.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yjmob_datfra['Timestamp'] = time_df[DAY] + '-' + time_df[TIME]


,uid,d,t,x,y,Timestamp
0,0,0,1,79,86,1900-01-01-00:30:00
1,0,0,2,79,86,1900-01-01-01:00:00
2,0,0,8,77,86,1900-01-01-04:00:00
3,0,0,9,77,86,1900-01-01-04:30:00
4,0,0,19,81,89,1900-01-01-09:30:00
...,...,...,...,...,...,...
999995,814,35,4,116,57,1900-02-05-02:00:00
999996,814,35,5,116,57,1900-02-05-02:30:00
999997,814,35,7,116,57,1900-02-05-03:30:00
999998,814,35,8,116,57,1900-02-05-04:00:00


In [ ]:
def yjmob100k_normalize(yjmob_dataframe):
    """
    Normalize the YJMob100K dataset by standardizing column names,
    computing a real timestamp, and casting to normalized datatypes.
    """
    yjmob_dataframe = compute_yjmob_timestamp(yjmob_dataframe)

    # Rename and drop in a single line to avoid extra copies
    yjmob_dataframe = (
        yjmob_dataframe
        .drop(columns=["d", "t"])
        .rename(columns={
            "uid": "DeviceID",
            "x": "Latitude",
            "y": "Longitude",
            "Timestamp": "Timestamp"  
        })
    )

    # Cast to final datatypes (you must define `normalized_datatypes`)
    return yjmob_dataframe.astype(normalized_datatypes)

yjmob_df_normalized = yjmob100k_normalize(yjmob_df)
yjmob_df_normalized

,DeviceID,Latitude,Longitude,Timestamp
0,0,79.0,86.0,1900-01-01-00:30:00
1,0,79.0,86.0,1900-01-01-01:00:00
2,0,77.0,86.0,1900-01-01-04:00:00
3,0,77.0,86.0,1900-01-01-04:30:00
4,0,81.0,89.0,1900-01-01-09:30:00
...,...,...,...,...
111535170,99999,119.0,77.0,1900-03-16-19:00:00
111535171,99999,132.0,94.0,1900-03-16-19:30:00
111535172,99999,124.0,105.0,1900-03-16-20:00:00
111535173,99999,121.0,107.0,1900-03-16-20:30:00


In [15]:
SAVE_PATH_YJMOB100K = Path(r"PreprocessedData/YJMob100Kv3") / 'yjmob100k_normalizedcols.pkl'

begin = time()
yjmob_df_normalized.to_pickle(path = SAVE_PATH_YJMOB100K, compression=None)
end = time()
print(f"Saving duration: {(end-begin):.2f} seconds.")

Saving duration: 58.19 seconds.


In [20]:
all_yjmob100k_df = pd.read_pickle('PreprocessedData/0NormalizedCols/yjmob100k.pkl')
all_yjmob100k_df.to_csv('PreprocessedData/0NormalizedCols/yjmob100k.csv', index=False)

# Shanghai Kaggle

In [14]:
def process_shanghai_stationary_fast(df, interval_minutes=30):
    # Rename and convert columns
    df = df.rename(columns={
        'latitude': 'Latitude',
        'longitude': 'Longitude',
        'user id': 'DeviceID'
    }).copy()

    df['DeviceID'] = df['DeviceID'].astype(str)
    df['Latitude'] = df['Latitude'].astype('float64')
    df['Longitude'] = df['Longitude'].astype('float64')
    df['start time'] = pd.to_datetime(df['start time'])
    df['end time'] = pd.to_datetime(df['end time'])

    # Compute stationarity time in minutes
    df['stationarity_time'] = (df['end time'] - df['start time']).dt.total_seconds() / 60

    output_rows = []

    # Append one row with end time for all
    base_rows_start = df[['DeviceID', 'Latitude', 'Longitude', 'start time']].copy()
    base_rows_start = base_rows_start.rename(columns={'start time': 'Timestamp'})
    base_rows_end = df[['DeviceID', 'Latitude', 'Longitude', 'end time']].copy()
    base_rows_end = base_rows_end.rename(columns={'end time': 'Timestamp'})
    output_rows.append(base_rows_start)
    print("done1")
    output_rows.append(base_rows_end)
    print("done2")

    # Handle rows with long stationarity
    long_df = df[df['stationarity_time'] > interval_minutes]
    print(len(long_df))
    for idx, row in long_df.iterrows():
        start = row['start time']
        end = row['end time']
        device_id = row['DeviceID']
        lat = row['Latitude']
        lon = row['Longitude']

        # Create intermediate timestamps
        intermediate_times = pd.date_range(
            start=start + pd.Timedelta(minutes=interval_minutes),
            end=end,
            freq=f'{interval_minutes}min'
        )

        for ts in intermediate_times:
            output_rows.append(pd.DataFrame([{
                'Timestamp': ts,
                'Latitude': lat,
                'Longitude': lon,
                'DeviceID': device_id
            }]))

    # Concatenate all together
    result_df = pd.concat(output_rows, ignore_index=True)
    print("done3")

    # Final dtype enforcement
    return result_df.astype({
        'DeviceID': 'str',
        'Latitude': 'float64',
        'Longitude': 'float64',
        'Timestamp': 'datetime64[ns]'
    })


In [ ]:
path_to_folder = Path(r"Raw_Datasets/Shanghai_Kaggle")

excel_files = list(path_to_folder.glob("*.xlsx"))

for file in excel_files:
    print(f"Reading {file.name}...")
    shanghai_df = pd.read_excel(file, engine='openpyxl')
    
    shanghai_df['start time'] = pd.to_datetime(shanghai_df['start time'])
    shanghai_df['end time'] = pd.to_datetime(shanghai_df['end time'])
    shanghai_df['stationarity_time'] = (shanghai_df['end time'] - shanghai_df['start time']).dt.total_seconds() / 60
    shanghai_df = shanghai_df.dropna()
    shanghai_df_processed = process_shanghai_stationary_fast(shanghai_df, interval_minutes=15)
    
    SAVE_PATH_SHANGHAI_FILE = Path(r"PreprocessedData/ShanghaiKaggle") / (file.stem + '.pkl')
    begin = time()
    shanghai_df_processed.to_pickle(path = SAVE_PATH_SHANGHAI_FILE)
    end = time()
    print(f"Saving duration: {(end-begin):.2f} seconds.")

Reading data_11.111.15.xlsx...
done1
done2
215534


In [19]:
all_shanghai_df = pd.DataFrame()
folder_path = "PreprocessedData/ShanghaiKaggle"
for file in os.listdir(folder_path):
    df = pd.read_pickle(folder_path+'/'+file)
    all_shanghai_df = pd.concat([all_shanghai_df, df])
all_shanghai_df.to_pickle('PreprocessedData/NormalizedCols/shanghaikaggle.pkl')
all_shanghai_df.to_csv('PreprocessedData/NormalizedCols/shanghaikaggle.csv', index=False)